# 🧭 Decoding 450,000+ Job Postings: The Empirical Labor Market Intelligence Report
### Beyond Generic Data-to-Graph Generation: Seniority Gatekeeping, Ghost Jobs, Pay Transparency, and High-ROI Skill Stacks across 7,500+ Employers

**Live Search & Exploration Portal:** [chandranshb.github.io/roleBase](https://chandranshb.github.io/roleBase/)  
**Source Code & Automated Daily Crawler:** [github.com/chandranshB/roleBase](https://github.com/chandranshB/roleBase)  
**Dataset:** [Job Listings: Open Jobs in Every Field (Daily)](https://www.kaggle.com/datasets/chandranshbinjola/job-listings)

---

### 💡 Executive Briefing: Why This Notebook Exists
Most job market analyses suffer from two fatal flaws:
1. **Tech-Blinkered Bias:** They scrape a few thousand software engineering jobs from Hacker News or boutique tech boards and pretend it represents the entire economy.
2. **Superficial Aggregation:** They produce generic bar charts of *"Top 10 Job Titles"* without asking the uncomfortable questions that job seekers and analysts actually care about.

This report analyzes **over 450,000 active, verified job postings from 7,500+ employers across 20+ industries**—sourced directly from employer Applicant Tracking Systems (Greenhouse, Lever, Ashby, Workday, Oracle, Phenom, SmartRecruiters, and more).

#### 🎯 Key Investigative Themes in This Report:
* **The "Missing Middle" & Seniority Gatekeeping:** Why mid-level roles command 60%+ of the market while true entry-level pipelines face severe compression.
* **The "Junior" Experience Inflation Trap:** How many "Junior" or "Associate" roles secretly demand 3 to 5+ years of experience, and which industries are the worst offenders.
* **Anatomy of Ghost Jobs:** Quantifying the prevalence of phantom requisitions, evergreen resume harvesters, and postings open for 6–12+ months.
* **The Pay Transparency Divide:** Why salary disclosure depends almost entirely on the ATS engine (Ashby/Lever/Greenhouse vs. Workday/Oracle) and where the "Remote Pay Premium" actually exists.
* **Skill Valuation & Stacking:** Separating high-frequency "commodity" skills from exponential "compensation multipliers" ($110k vs. $245k median pay).
* **Posting Lifespan & Hiring Velocity:** Quantifying the critical **48-to-72-hour application window** before a listing becomes functionally dead.
* **Interactive Job Seeker Navigator:** An executable search function to extract high-pay, verified, non-ghost postings tailored to your preferences.


In [ ]:
import glob
import os
import warnings
from collections import Counter
from itertools import combinations
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings('ignore')

# Aesthetic styling for publication-grade charts
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Helvetica']
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['figure.autolayout'] = True
plt.rcParams['figure.dpi'] = 110

# Dual-path data loader: seamless on Kaggle and local development environments (supports Parquet and CSV)
def locate_file(filename):
    base = filename.rsplit(".", 1)[0]
    candidates = [f"{base}.parquet", f"{base}.csv", filename] if "." in filename else [f"{filename}.parquet", f"{filename}.csv"]
    for cand in candidates:
        kaggle_matches = glob.glob(f"/kaggle/input/**/{cand}", recursive=True)
        if kaggle_matches:
            return kaggle_matches[0]
        local_matches = glob.glob(f"kaggle/{cand}") or glob.glob(f"../kaggle/{cand}") or glob.glob(f"data/{cand}")
        if local_matches:
            return local_matches[0]
    raise FileNotFoundError(f"Could not locate {filename} (as parquet or csv) in Kaggle or local workspace paths.")

def load_dataset(filename, parse_dates=None):
    path = locate_file(filename)
    if path.endswith(".parquet"):
        df = pd.read_parquet(path)
        if parse_dates:
            for c in parse_dates:
                if c in df.columns and not pd.api.types.is_datetime64_any_dtype(df[c]):
                    df[c] = pd.to_datetime(df[c], errors="coerce")
        return df
    return pd.read_csv(path, low_memory=False, parse_dates=parse_dates)

# Load primary datasets
jobs_path = locate_file("jobs.parquet")
print(f"Loading primary dataset from: {jobs_path} ...")
df = load_dataset("jobs", parse_dates=["posted", "first_seen", "closed_at"])

open_jobs = df[df.closed_at.isna()].copy()
closed_jobs = df[df.closed_at.notna()].copy()
closed_jobs['days_open'] = (closed_jobs.closed_at - closed_jobs.first_seen).dt.days

# Stated USD pay subset
usd_jobs = open_jobs[(open_jobs.pay_currency == "USD") & open_jobs.pay_min.notna() & open_jobs.pay_max.notna()].copy()
usd_jobs['mid_pay'] = (usd_jobs.pay_min + usd_jobs.pay_max) / 2
usd_jobs['pay_spread_pct'] = (usd_jobs.pay_max - usd_jobs.pay_min) / usd_jobs.pay_min * 100

print("\n" + "="*70)
print("[LABOR MARKET DATASET VITAL SIGNS]")
print("="*70)
print(f"- Total Tracked Postings : {len(df):,}")
print(f"- Active Open Postings   : {len(open_jobs):,} ({len(open_jobs)/len(df):.1%})")
print(f"- Verified Closed Postings: {len(closed_jobs):,} (benchmark for posting lifespan)")
print(f"- Unique Employers       : {df.company.nunique():,}")
print(f"- Industries / Domains   : {df.category.nunique():,}")
print(f"- Postings with USD Pay  : {len(usd_jobs):,} ({len(usd_jobs)/len(open_jobs):.1%})")
print(f"- Date Range Tracked     : {df.first_seen.min().strftime('%Y-%m-%d')} to {df.first_seen.max().strftime('%Y-%m-%d')}")
print("="*70)


## 1. The Macro Hiring Landscape: Industry Distribution & The "Missing Middle"

Unlike datasets restricted to Silicon Valley tech startups, this dataset reflects the real, multifaceted economy. While **Engineering (75k+)** and **Data (16k+)** have massive hiring presences, traditional powerhouses like **Retail (48k+)**, **Sales (45k+)**, **Healthcare (37k+)**, **Finance (31k+)**, and **Operations (30k+)** constitute over 60% of open requisitions.

### The Seniority Distribution Paradox
Notice the extreme concentration at the **Mid-Level (66.4%)**:
* Companies overwhelmingly demand autonomous contributors who require zero training or onboarding hand-holding.
* **Junior roles represent only 10.0%** of open postings, and **Internships represent a mere 3.3%**.
* This structural squeeze creates the modern hiring crunch: the market is desperate for experienced talent, but reluctant to train the entry-level talent required to build tomorrow's workforce.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: Top Industry Domains
cat_counts = open_jobs.category.value_counts().head(14).sort_values()
cat_pct = (cat_counts / len(open_jobs) * 100)
colors_cat = sns.color_palette("mako", len(cat_counts))

bars1 = axes[0].barh(cat_counts.index, cat_counts.values, color=colors_cat, edgecolor='none', height=0.68)
axes[0].set_title("Open Positions by Professional Field (Top 14)", pad=12)
axes[0].set_xlabel("Number of Open Postings")
for bar, pct in zip(bars1, cat_pct):
    axes[0].text(bar.get_width() + 1000, bar.get_y() + bar.get_height()/2, 
                 f"{bar.get_width():,} ({pct:.1f}%)", va='center', fontsize=9, fontweight='medium', color='#333333')
axes[0].set_xlim(0, cat_counts.max() * 1.22)

# Panel 2: The Seniority Hierarchy
level_order = ["intern", "junior", "mid", "senior", "lead", "staff+", "manager+"]
level_counts = open_jobs.level.value_counts().reindex(level_order).fillna(0)
level_pct = (level_counts / len(open_jobs) * 100)
colors_level = ['#2a9d8f', '#e76f51', '#264653', '#e9c46a', '#f4a261', '#457b9d', '#1d3557']

bars2 = axes[1].barh([l.upper() for l in level_order], level_counts.values, color=colors_level, edgecolor='none', height=0.68)
axes[1].set_title("The Seniority Hierarchy (Open Requisitions)", pad=12)
axes[1].set_xlabel("Number of Open Postings")
for bar, pct in zip(bars2, level_pct):
    axes[1].text(bar.get_width() + 4000, bar.get_y() + bar.get_height()/2, 
                 f"{bar.get_width():,} ({pct:.1f}%)", va='center', fontsize=9, fontweight='medium', color='#333333')
axes[1].set_xlim(0, level_counts.max() * 1.25)

plt.tight_layout()
plt.show()


## 2. The "Fresher" Reality: Auditing Experience Inflation & Junior Gatekeeping

One of the most persistent frustrations among early-career professionals is applying to a posting titled **"Junior Analyst"** or **"Associate Engineer"**, only to find the qualification section demanding:
> *"3-5 years of direct enterprise experience required."*

To audit this empirically, we cross-reference positions categorized with seniority level `junior` against the explicit `years_experience` demanded in the job specification text.

### Key Empirical Findings:
1. **The Worst Offenders:** **Legal (41.0%)**, **Marketing (39.5%)**, and **Finance (33.7%)** exhibit rampant title inflation—over a third of their "junior" postings demand 3 or more years of experience!
2. **Tech & Data Gatekeeping:** In **Data** and **Engineering**, the inflation rate sits around **10-12%**, but the baseline bar remains high: even entry-level tech roles demand multi-technology competency stacks (Python + SQL + Cloud).
3. **"Fresher Havens" vs. "Gatekeepers":** Below, we uncover the enterprise employers with 50+ open roles that genuinely welcome 0–2 years of experience versus those that inflate junior job titles.


In [ ]:
# Analyze Experience Inflation among Junior roles
juniors = open_jobs[open_jobs.level == "junior"].copy()
j_with_years = juniors.dropna(subset=["years_experience"])

j_cat_stats = juniors.groupby("category").agg(
    total_junior=("title", "count"),
    junior_stating_years=("years_experience", lambda s: s.notna().sum()),
    asking_3plus=("years_experience", lambda s: (s >= 3).sum()),
    asking_5plus=("years_experience", lambda s: (s >= 5).sum())
)
j_cat_stats["inflation_rate_pct"] = (j_cat_stats["asking_3plus"] / j_cat_stats["junior_stating_years"] * 100).round(1)
j_top_inflated = j_cat_stats[j_cat_stats.junior_stating_years >= 40].sort_values("inflation_rate_pct", ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: Inflation Rate by Category
bars = axes[0].barh(j_top_inflated.index, j_top_inflated["inflation_rate_pct"], color='#e63946', height=0.65)
axes[0].set_title("Junior Experience Inflation: % of Junior Roles Asking for 3+ Years", pad=12)
axes[0].set_xlabel("Experience Inflation Rate (%) [Asking >= 3 Yrs Experience]")
for bar in bars:
    axes[0].text(bar.get_width() + 0.8, bar.get_y() + bar.get_height()/2, 
                 f"{bar.get_width():.1f}%", va='center', fontsize=9, fontweight='bold', color='#333333')
axes[0].set_xlim(0, j_top_inflated["inflation_rate_pct"].max() * 1.2)

# Panel 2: Distribution of Explicit Years Asked Across Seniority Tiers
exp_subset = open_jobs.dropna(subset=["years_experience"]).copy()
exp_subset = exp_subset[exp_subset.level.isin(["junior", "mid", "senior", "lead", "staff+"])]
sns.boxplot(data=exp_subset, x="level", y="years_experience", order=["junior", "mid", "senior", "lead", "staff+"],
            palette="Set2", showmeans=True, meanprops={"marker":"o", "markerfacecolor":"white", "markeredgecolor":"black"}, ax=axes[1])
axes[1].set_title("Actual Explicit Years of Experience Demanded per Level", pad=12)
axes[1].set_xlabel("Seniority Tier")
axes[1].set_ylabel("Required Years of Experience")
axes[1].set_ylim(0, 16)

plt.tight_layout()
plt.show()

print(f"[TOTAL AUDITED JUNIOR POSITIONS]: {len(juniors):,}")
print(f"- Junior positions explicitly stating experience: {len(j_with_years):,}")
print(f"- Junior positions demanding 3+ years: {(juniors.years_experience >= 3).sum():,} ({(juniors.years_experience >= 3).sum()/len(j_with_years):.1%})")
print(f"- Junior positions demanding 5+ years: {(juniors.years_experience >= 5).sum():,}")


In [ ]:
# Load company-level benchmarks
comp_path = locate_file("jobs_by_company.parquet")
companies = load_dataset("jobs_by_company")

# Filter for companies with substantial hiring volume (>= 50 open roles)
big_companies = companies[companies.open_roles >= 50].copy()

# Table 1: Top Fresher Havens (highest share of roles open to 0-2 years)
top_fresher = big_companies.nlargest(8, "fresher_share")[
    ["company", "open_roles", "fresher_roles", "fresher_share", "junior_asking_3plus"]
].copy()
top_fresher["fresher_share"] = (top_fresher["fresher_share"] * 100).round(1).astype(str) + "%"

# Table 2: Top Junior Inflators (most junior roles asking for 3+ years)
top_inflators = big_companies.nlargest(8, "junior_asking_3plus")[
    ["company", "open_roles", "junior_asking_3plus", "years_stated_share"]
].copy()
top_inflators["years_stated_share"] = (top_inflators["years_stated_share"] * 100).round(1).astype(str) + "%"

print("[TOP FRESHER HAVENS] (50+ Open Roles with Highest 0-2 Year Accessibility):")
display(top_fresher)

print("\n[TOP TITLE INFLATORS] (Employers with the Most Junior Roles Demanding 3+ Years):")
display(top_inflators)


## 3. Anatomy of the "Ghost Job" Epidemic: Quantifying Phantom Requisitions

A **"Ghost Job"** is an active job posting that the employer has no immediate intention of filling. Common industry drivers include:
* **Perpetual Talent Pooling:** Keeping a generic requisition open 365 days a year to build a passive resume pipeline.
* **Investor & Market Optics:** Projecting rapid growth and financial health to competitors, clients, and investors.
* **HR Automation Neglect:** Forgetting to close or delist requisitions after headcount freezes or internal backfills.
* **Repost Loops:** Automated systems reposting identical templates dozens of times to maintain algorithmic top-of-feed placement.

### How Ghost Risk is Calculated:
In this dataset, each open posting receives a heuristic `ghost_score` based on deterministic audit triggers:
* **Stale Lifespan:** Open for $>6$ months (+2 points) or $>1$ year (+3 points).
* **Massive Duplicate Clusters:** Postings sharing identical title, team, and location cloned $\ge 5$ times.
* **Talent Pool Phrasing:** Explicit disclaimers stating "this is not an active role" or "talent community submission".

> **The Job Hunter's Heuristic:** Postings with a `ghost_score` $\ge 2$ have a significantly lower probability of review. Candidates should prioritize fresh listings ($<14$ days old) with unique requisition IDs.


In [ ]:
# Analyze Ghost Postings
ghost_subset = open_jobs[open_jobs.ghost_score >= 1].copy()
high_ghost = open_jobs[open_jobs.ghost_score >= 2].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# Panel 1: Top Ghost Audit Triggers
trigger_counts = Counter()
for reason_str in open_jobs.ghost_reasons.dropna():
    for r in reason_str.split("; "):
        if "identical postings" in r:
            trigger_counts["High-frequency duplicate postings"] += 1
        else:
            trigger_counts[r.capitalize()] += 1

triggers_s = pd.Series(trigger_counts).sort_values(ascending=True).tail(7)
axes[0].barh(triggers_s.index, triggers_s.values, color='#8338ec', height=0.62)
axes[0].set_title("Primary Ghost Job Heuristic Triggers Detected", pad=12)
axes[0].set_xlabel("Number of Affected Open Postings")
for idx, val in enumerate(triggers_s.values):
    axes[0].text(val + 300, idx, f"{val:,}", va='center', fontsize=9, fontweight='bold', color='#333333')
axes[0].set_xlim(0, triggers_s.max() * 1.2)

# Panel 2: Ghost Posting Share by Category
ghost_by_cat = open_jobs.groupby("category").agg(
    total=("title", "count"),
    high_ghost=("ghost_score", lambda s: (s >= 2).sum())
)
ghost_by_cat["ghost_rate_pct"] = (ghost_by_cat["high_ghost"] / ghost_by_cat["total"] * 100).round(2)
top_ghost_cats = ghost_by_cat[ghost_by_cat.total >= 1000].sort_values("ghost_rate_pct", ascending=True).tail(10)

bars_g = axes[1].barh(top_ghost_cats.index, top_ghost_cats["ghost_rate_pct"], color='#ff006e', height=0.62)
axes[1].set_title("Ghost Posting Rate by Field (% with Ghost Score >= 2)", pad=12)
axes[1].set_xlabel("High Ghost Risk Rate (%)")
for bar in bars_g:
    axes[1].text(bar.get_width() + 0.15, bar.get_y() + bar.get_height()/2, 
                 f"{bar.get_width():.2f}%", va='center', fontsize=9, fontweight='bold', color='#333333')
axes[1].set_xlim(0, top_ghost_cats["ghost_rate_pct"].max() * 1.25)

plt.tight_layout()
plt.show()

print("[GHOST REQUISITION AUDIT SUMMARY]:")
print(f"- Total active postings with ghost warning triggers (Score >= 1): {len(ghost_subset):,} ({len(ghost_subset)/len(open_jobs):.1%})")
print(f"- Severe ghost risk requisitions (Score >= 2): {len(high_ghost):,} ({len(high_ghost)/len(open_jobs):.1%})")
print(f"- Postings open for over 6 months: {(open_jobs.ghost_reasons.str.contains('open for over 6 months', na=False)).sum():,}")
print(f"- Postings open for over 1 full year: {(open_jobs.ghost_reasons.str.contains('open for over a year', na=False)).sum():,}")


## 4. Compensation Intelligence: The Transparency Divide & Pay Curves

Salary transparency is mandated by law in jurisdictions like California, New York, Colorado, and Washington. However, across the global labor market, our data reveals an astonishing disparity:

### 1. The ATS Transparency Divide
* Modern, high-growth ATS platforms enforce structured salary fields:
  * **Ashby:** **52.8%** of postings disclose compensation!
  * **Lever:** **46.7%** disclose compensation!
  * **Greenhouse:** **38.6%** disclose compensation!
* Legacy enterprise ERP suites (**Workday, Oracle Recruiting, Phenom, SmartRecruiters**): **0.0% structured pay disclosure** in public API feeds.
* *Takeaway for job seekers:* If you want upfront salary clarity before applying, focus your search on employers using Ashby, Lever, and Greenhouse.

### 2. The Career Progression Curve & The "5-Year Jump"
When plotting median USD pay against required years of experience, compensation scales predictably—until **Year 5**:
* At 5 years, median compensation surges by **+$24,000** (from $160.8k to $185.0k) and hiring volume triples. Year 5 represents the universal industry threshold between supervised executor and autonomous contributor.

### 3. The Remote Pay Reality: Premium or Penalty?
Does remote work pay less due to geographic cost-of-living adjustments?
* **Across Mid, Senior, and Lead tiers, Remote work commands a $4,500 to $20,000 PREMIUM.**
* Why? Companies offering remote work compete in an open, national/global talent market. They must pay top-of-market rates to attract premier talent, rather than benchmarking against local commuter wages.
* **The exception:** Junior remote roles pay slightly less ($102.5k vs. $120.2k), as employers often source entry-level remote talent from lower-cost regions.


In [ ]:
# Analyze ATS Pay Transparency
ats_stats = open_jobs.groupby("source").agg(
    total_postings=("title", "count"),
    disclosing_pay=("pay_min", lambda s: s.notna().sum())
)
ats_stats["disclosure_rate_pct"] = (ats_stats["disclosing_pay"] / ats_stats["total_postings"] * 100).round(1)
top_ats = ats_stats[ats_stats.total_postings >= 1000].sort_values("disclosure_rate_pct", ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# Panel 1: ATS Pay Transparency Divide
bars_ats = axes[0].barh(top_ats.index, top_ats["disclosure_rate_pct"], color='#2a9d8f', height=0.62)
axes[0].set_title("Salary Transparency by ATS Engine (% Disclosing Pay)", pad=12)
axes[0].set_xlabel("Pay Transparency Rate (%)")
for bar in bars_ats:
    axes[0].text(bar.get_width() + 1.0, bar.get_y() + bar.get_height()/2, 
                 f"{bar.get_width():.1f}%", va='center', fontsize=9, fontweight='bold', color='#333333')
axes[0].set_xlim(0, 65)

# Panel 2: Salary Range Spread Distribution (Testing Range Integrity)
valid_spread = usd_jobs[usd_jobs.pay_spread_pct.between(0, 150)]
sns.histplot(valid_spread.pay_spread_pct, bins=35, kde=True, color='#457b9d', ax=axes[1])
axes[1].axvline(valid_spread.pay_spread_pct.median(), color='#e63946', linestyle='--', linewidth=2, 
                label=f"Median Spread: {valid_spread.pay_spread_pct.median():.1f}%")
axes[1].set_title("Salary Range Spread Distribution [(Max - Min) / Min * 100]", pad=12)
axes[1].set_xlabel("Salary Range Width (%)")
axes[1].set_ylabel("Job Postings")
axes[1].legend(loc="upper right", frameon=True)

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# Panel 1: The Career Pay Curve (Years of Experience vs Median Pay)
exp_pay = usd_jobs.dropna(subset=["years_experience"]).groupby("years_experience").agg(
    median_pay=("mid_pay", "median"),
    job_count=("mid_pay", "count")
).query("job_count >= 20 and years_experience <= 12").reset_index()

axes[0].plot(exp_pay["years_experience"], exp_pay["median_pay"] / 1000, marker='o', linewidth=2.8, markersize=8, color='#1d3557')
axes[0].set_title("The Career Pay Progression Curve (USD)", pad=12)
axes[0].set_xlabel("Explicit Minimum Years of Experience Demanded")
axes[0].set_ylabel("Median Annual Compensation ($k USD)")
axes[0].axvspan(4.5, 5.5, color='#e9c46a', alpha=0.3, label="The '5-Year Senior Inflection Jump'")
for x, y in zip(exp_pay["years_experience"], exp_pay["median_pay"] / 1000):
    axes[0].annotate(f"${y:.0f}k", (x, y), textcoords="offset points", xytext=(0, 9), ha='center', fontsize=8.5, fontweight='bold')
axes[0].legend(loc="lower right")

# Panel 2: Remote vs On-Site Pay by Seniority Tier
level_order_pay = ["junior", "mid", "senior", "lead", "staff+", "manager+"]
remote_pay = usd_jobs[usd_jobs.level.isin(level_order_pay)].groupby(["level", "remote"]).agg(
    median_pay=("mid_pay", "median"),
    count=("mid_pay", "count")
).unstack()

remote_df = pd.DataFrame({
    'On-Site / Hybrid': remote_pay['median_pay'][False] / 1000,
    'Fully Remote': remote_pay['median_pay'][True] / 1000
}).reindex(level_order_pay)

x = np.arange(len(level_order_pay))
width = 0.35
rects1 = axes[1].bar(x - width/2, remote_df['On-Site / Hybrid'], width, label='On-Site / Hybrid', color='#457b9d')
rects2 = axes[1].bar(x + width/2, remote_df['Fully Remote'], width, label='Fully Remote', color='#2a9d8f')

axes[1].set_title("Median Compensation: Remote vs. On-Site by Seniority", pad=12)
axes[1].set_xlabel("Seniority Tier")
axes[1].set_ylabel("Median Annual Compensation ($k USD)")
axes[1].set_xticks(x)
axes[1].set_xticklabels([l.upper() for l in level_order_pay])
axes[1].legend(loc="upper left")
axes[1].set_ylim(0, 280)

# Add value labels
for rect in rects1:
    h = rect.get_height()
    axes[1].annotate(f"${h:.0f}k", xy=(rect.get_x() + rect.get_width()/2, h), xytext=(0, 3),
                     textcoords="offset points", ha='center', va='bottom', fontsize=8)
for rect in rects2:
    h = rect.get_height()
    axes[1].annotate(f"${h:.0f}k", xy=(rect.get_x() + rect.get_width()/2, h), xytext=(0, 3),
                     textcoords="offset points", ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.show()


## 5. Skill Valuation: Commodity Baseline Tools vs. High-ROI Multipliers

Not all skills carry equal weight on a resume. By cross-referencing normalized skill mentions against verified compensation, we can divide the skill ecosystem into three clear tiers:

1. **High-Frequency Commodities (Table Stakes):**
   * *Examples:* Customer Service, Inventory Management, MS Office, Basic Excel.
   * *Economic Reality:* Widespread demand, but lower pricing power ($70k to $133k median pay).
2. **Core Operational Workhorses:**
   * *Examples:* Python, AWS, SQL, Docker, Java, Agile.
   * *Economic Reality:* High volume and strong upper-middle-class compensation ($170k to $210k).
3. **High-Leverage Multipliers (Scarcity Premium):**
   * *Examples:* **PyTorch ($245k)**, **TensorFlow ($240k)**, **Rust ($235k)**, **Redis ($226k)**, **LLM / GenAI ($225k)**, **C++ ($225k)**, **Kubernetes ($220k)**.
   * *Economic Reality:* Specialized technical mastery that directly impacts production infrastructure and proprietary AI workflows.

### The Skill Stacking Principle:
High earners rarely know just one tool. They stack complementary competencies:
* **The AI / ML Production Stack:** `Python` + `PyTorch` + `AWS` + `Kubernetes`
* **The Modern Data Platform Stack:** `Python` + `SQL` + `Snowflake` / `Airflow` + `dbt`
* **The Enterprise Cloud Stack:** `Go` / `Rust` + `Docker` + `Terraform` + `Kubernetes`


In [ ]:
# Explode skills to calculate compensation per skill
usd_with_skills = usd_jobs.dropna(subset=["skills"]).copy()
usd_with_skills["skill_list"] = usd_with_skills.skills.str.split(";")
exploded_skills = usd_with_skills.explode("skill_list")

skill_stats = exploded_skills.groupby("skill_list").agg(
    job_count=("mid_pay", "count"),
    median_pay=("mid_pay", "median")
).query("job_count >= 40").sort_values("median_pay", ascending=False)

top_paying_skills = skill_stats.head(10)
bottom_paying_skills = skill_stats.tail(10)
spectrum = pd.concat([bottom_paying_skills, top_paying_skills])

fig, ax = plt.subplots(figsize=(14, 7))
bar_colors = ['#d90429' if p < 150000 else '#1b4332' for p in spectrum['median_pay']]
bars_sk = ax.barh(spectrum.index, spectrum['median_pay'] / 1000, color=bar_colors, height=0.68)

ax.set_title("The Skill Compensation Spectrum: Top 10 High-Leverage vs. 10 Commodity Skills (USD)", pad=14)
ax.set_xlabel("Median Annual Compensation ($k USD)")
ax.axvline(180, color='#666666', linestyle=':', label='Market Benchmark ($180k)')

for bar in bars_sk:
    w = bar.get_width()
    ax.text(w + 3, bar.get_y() + bar.get_height()/2, f"${w:.0f}k", 
            va='center', fontsize=9, fontweight='bold', color='#333333')

ax.set_xlim(0, 275)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()


In [ ]:
# Analyze Skill Co-occurrences across open postings
skill_pairs = Counter()
for s in open_jobs.skills.dropna():
    sk_list = sorted(list(set(s.split(";"))))
    if len(sk_list) > 1:
        for pair in combinations(sk_list, 2):
            skill_pairs[pair] += 1

top_pairs = pd.DataFrame([
    {"skill_A": p[0], "skill_B": p[1], "frequency": count, "pair": f"{p[0]} + {p[1]}"}
    for p, count in skill_pairs.most_common(14)
])

fig, ax = plt.subplots(figsize=(14, 6))
bars_pair = ax.barh(top_pairs["pair"][::-1], top_pairs["frequency"][::-1], color='#4361ee', height=0.68)
ax.set_title("Top Skill Co-occurrences Across Active Postings (The Power Combinations)", pad=12)
ax.set_xlabel("Number of Postings Requiring Both Skills Simultaneously")

for bar in bars_pair:
    w = bar.get_width()
    ax.text(w + 150, bar.get_y() + bar.get_height()/2, f"{w:,}", 
            va='center', fontsize=9, fontweight='medium', color='#333333')

ax.set_xlim(0, top_pairs["frequency"].max() * 1.18)
plt.tight_layout()
plt.show()


## 6. Hiring Velocity: Survival Analysis of Job Postings & The 48-Hour Window

Job seekers frequently apply to postings that have been floating on job boards for 30, 45, or 60 days, wondering why they never hear back.

By analyzing **over 21,000 verified closed job postings** (where both `first_seen` and `closed_at` were captured by the daily crawler), we can calculate the empirical **posting survival curve**:

### ⏱️ The Empirical Truth About Posting Lifespan:
* **The 48-Hour Threshold:** **25% of jobs close within 2 days.**
* **The 6-Day Half-Life:** **50% of closed positions are delisted within 6 calendar days.**
* **The 7-Day Cliff:** **75% of positions close within 7 days.**
* **The 10-Day Horizon:** By day 10, **90% of closed postings are gone.**

> **Actionable Recommendation:** If a job posting is more than 7 days old, the candidate pool is almost certainly closed, shortlisted, or in final-round interviews. **Apply within the first 48 to 72 hours of listing** to ensure your application is actually evaluated by human reviewers.


In [ ]:
# Lifespan analysis of closed positions
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# Panel 1: Cumulative Closure Curve (Posting Survival Rate)
closed_valid = closed_jobs[closed_jobs.days_open.between(0, 30)].copy()
sorted_days = np.sort(closed_valid.days_open)
cumulative_pct = np.arange(1, len(sorted_days) + 1) / len(sorted_days) * 100

axes[0].plot(sorted_days, cumulative_pct, color='#d90429', linewidth=2.8)
axes[0].set_title("Cumulative Job Delisting Curve (% Closed by Day N)", pad=12)
axes[0].set_xlabel("Days Since First Detected")
axes[0].set_ylabel("Cumulative % of Jobs Closed")
axes[0].axvline(2, color='#666666', linestyle='--', label="25th Percentile: 2 Days")
axes[0].axvline(6, color='#e76f51', linestyle='--', label="Median (Half-Life): 6 Days")
axes[0].axvline(7, color='#264653', linestyle='--', label="75th Percentile: 7 Days")
axes[0].axvline(10, color='#1d3557', linestyle='--', label="90th Percentile: 10 Days")
axes[0].set_xlim(0, 20)
axes[0].set_ylim(0, 105)
axes[0].legend(loc="lower right", frameon=True)

# Panel 2: Median Days to Close by Industry Domain
cat_velocity = closed_jobs.groupby("category").agg(
    median_days=("days_open", "median"),
    closed_count=("days_open", "count")
).query("closed_count >= 150").sort_values("median_days", ascending=True)

bars_vel = axes[1].barh(cat_velocity.index, cat_velocity["median_days"], color='#2a9d8f', height=0.65)
axes[1].set_title("Median Days a Posting Stays Active by Field", pad=12)
axes[1].set_xlabel("Median Days Open Before Delisting")
for bar in bars_vel:
    w = bar.get_width()
    axes[1].text(w + 0.15, bar.get_y() + bar.get_height()/2, f"{w:.0f} days", 
                 va='center', fontsize=9, fontweight='bold', color='#333333')
axes[1].set_xlim(0, 9)

plt.tight_layout()
plt.show()

print("[EMPIRICAL POSTING LIFESPAN QUANTILES]:")
quantiles = closed_jobs.days_open.quantile([0.10, 0.25, 0.50, 0.75, 0.90, 0.95])
for q, v in quantiles.items():
    print(f"- {int(q*100)}th Percentile: {v:.0f} days")


## 7. Shifting Skill Trajectories: Longitudinal Skill Pulse

Using `skill_demand_daily.csv`, we track how the percentage share of skill-tagged open jobs evolves over time. This daily telemetry provides an early-warning signal for emerging framework adoption and shifting employer requirements.


In [ ]:
skill_trend_path = locate_file("skill_demand_daily.parquet")
trend_df = load_dataset("skill_demand_daily", parse_dates=["date"])

# Identify the top 8 skills by share on the latest date
latest_date = trend_df.date.max()
top_recent_skills = trend_df[trend_df.date == latest_date].nlargest(8, "share_pct")["skill"].tolist()

trend_pivot = trend_df[trend_df.skill.isin(top_recent_skills)].pivot(
    index="date", columns="skill", values="share_pct"
)

fig, ax = plt.subplots(figsize=(14, 5.5))
trend_pivot.plot(ax=ax, marker='o', linewidth=2.2, markersize=5)
ax.set_title("Daily Share of Skill-Tagged Open Requisitions (% Mentioning Skill)", pad=12)
ax.set_xlabel("Date (UTC)")
ax.set_ylabel("% of Tagged Postings")
ax.legend(title="Tracked Skill", bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()


## 8. Interactive Job Opportunity Navigator: Finding Verified, Non-Ghost Roles

To transform this notebook from a passive analytical report into an **actionable utility**, here is an interactive query tool. You can filter the 450,000+ open jobs to find verified opportunities that match your specific criteria—complete with direct company application URLs.


In [ ]:
def search_verified_jobs(
    df_jobs,
    category=None,
    level=None,
    skill_required=None,
    min_salary_usd=None,
    remote_only=False,
    max_ghost_score=1,
    fresher_only=False,
    top_n=10
):
    # Filter active job postings for verified, high-quality, actionable opportunities.
    # Start with active open positions
    res = df_jobs[df_jobs.closed_at.isna()].copy()
    
    # Filter ghost risk
    res = res[res.ghost_score.fillna(0) <= max_ghost_score]
    
    # Optional filters
    if category:
        res = res[res.category.str.lower() == category.lower()]
    if level:
        res = res[res.level.str.lower() == level.lower()]
    if remote_only:
        res = res[res.remote == True]
    if fresher_only:
        res = res[res.fresher_friendly == True]
    if skill_required:
        res = res[res.skills.str.contains(skill_required.lower(), na=False, case=False)]
    if min_salary_usd:
        res = res[(res.pay_currency == "USD") & (res.pay_max >= min_salary_usd)]
    
    # Sort by salary where available, then most recently seen
    res['sort_pay'] = res.pay_min.fillna(0)
    res = res.sort_values(by=['sort_pay', 'first_seen'], ascending=[False, False])
    
    display_cols = ['company', 'title', 'category', 'level', 'remote', 'location', 'pay_min', 'pay_max', 'ghost_score', 'url']
    results = res.head(top_n)[display_cols]
    
    print(f"[FOUND] {len(res):,} MATCHING OPPORTUNITIES (Showing top {len(results)})")
    return results

# Interactive Demo Search: High-Paying, Verified, Non-Ghost Remote Engineering / Data Roles
sample_results = search_verified_jobs(
    df,
    category="data",
    min_salary_usd=160000,
    remote_only=True,
    max_ghost_score=0,
    top_n=8
)

display(sample_results)


## 9. The Job Hunter's Empirical Playbook: 5 Evidence-Backed Rules

Based on our analysis of 450,000+ postings, here are the five strategic takeaways to maximize your application conversion rate:

1. **⏱️ Strike Within 72 Hours:**  
   *The data proves that 50% of closed postings disappear within 6 days.* Applying to postings older than 10 days is largely wasted effort. Set up automated daily keyword alerts and apply immediately.
2. **⚠️ Screen for "Ghost" Markers Before Investing Time in Cover Letters:**  
   Check when the role was posted. If the listing has been open for $>6$ months or is duplicated across dozens of identical requisitions, it is likely an evergreen resume bucket. Prioritize freshly verified requisitions with specific, non-templated team descriptions.
3. **🎯 Leverage the Ashby/Lever/Greenhouse Transparency Arbitrage:**  
   If you want clear salary ranges upfront, focus on companies utilizing modern venture ATS engines (Ashby, Lever, Greenhouse) where compensation disclosure exceeds 38–53%.
4. **🚀 Stack High-Leverage Multipliers onto Baseline Skills:**  
   Knowing `Python` or `SQL` puts you in the baseline talent pool. Pairing `Python` with `PyTorch`, `Kubernetes`, or `Rust` increases median compensation by **+$40,000 to +$65,000**.
5. **🏢 Target Genuine "Fresher Havens" for Early-Career Roles:**  
   Avoid applying blindly to "Junior" roles in Legal, Marketing, or Finance without reading the fine print—over 35% secretly demand 3+ years. Instead, cross-reference our *Fresher Havens* index to identify employers with a proven record of hiring candidates with 0–2 years of experience.

---

### 🔗 Explore More:
* **Interactive Web Explorer:** [chandranshb.github.io/roleBase](https://chandranshb.github.io/roleBase/)
* **GitHub Repository & Crawler:** [github.com/chandranshB/roleBase](https://github.com/chandranshB/roleBase)
* **Kaggle Dataset:** [Job Listings: Open Jobs in Every Field (Daily)](https://www.kaggle.com/datasets/chandranshbinjola/job-listings)
